# PicoRV32 CIM SoC — PYNQ 验证

**原理：**
- PicoRV32（RISC-V）在 PL 中自主运行 MNIST 推理
- 推理结果写入 Result BRAM（256B 双端口 BRAM）
- PS 通过 AXI GP 读取 Result BRAM，验证结果正确性

**Result BRAM 布局（0x4000_0000 起）：**
- word[0] = magic (0xC1AA0001 = 推理完成)
- word[1] = predicted class
- word[2] = expected label
- word[3] = match (1=correct)
- word[4..13] = fc2 logits (10 × INT8, sign-extended)

**前提：** 上传 `cim_rv32_soc.bit` + `cim_rv32_soc.hwh` 到 Jupyter 同目录

In [ ]:
from pynq import Overlay, MMIO
import numpy as np
import time

## 1. 加载 Bitstream

In [ ]:
ol = Overlay('cim_rv32_soc.bit')
print('Overlay loaded!')
print('PicoRV32 is now running...')

## 2. 读取 Result BRAM

In [ ]:
# Result BRAM mapped at 0x4000_0000, 4KB range
RES_BASE = 0x40000000
mmio = MMIO(RES_BASE, 0x1000)

MAGIC = 0xC1AA0001

# Poll for magic word (PicoRV32 writes it when inference is done)
print('Waiting for PicoRV32 to complete inference...')
t0 = time.time()
while True:
    magic = mmio.read(0x00)
    if magic == MAGIC:
        break
    if time.time() - t0 > 10:
        print(f'Timeout! magic=0x{magic:08x}')
        break
    time.sleep(0.1)

elapsed = time.time() - t0
print(f'Inference complete in {elapsed:.2f}s')

## 3. 读取推理结果

In [ ]:
pred     = mmio.read(0x04)
expected = mmio.read(0x08)
match    = mmio.read(0x0C)

logits = []
for i in range(10):
    v = mmio.read(0x10 + 4*i)
    # sign-extend from uint32 to int
    if v & 0x80000000:
        v = v - 0x100000000
    logits.append(v)

print(f'Predicted: {pred}')
print(f'Expected:  {expected}')
print(f'Match:     {"YES" if match else "NO"}')
print(f'Logits:    {logits}')

## 4. 验证：对比 Python golden model

In [ ]:
# Run the same INT8 inference in Python to cross-check
import sys, os

# Read test image and model data from small_mlp_data
# (upload small_mlp_data/ to PYNQ alongside the notebooks)
DATA_DIR = 'small_mlp_data'

def rhex_u32(path):
    with open(path) as f:
        return [int(l.strip(), 16) for l in f if l.strip()]

def rhex_u8(path):
    with open(path) as f:
        return [int(l.strip(), 16) & 0xFF for l in f if l.strip()]

# Read the same image that firmware uses (img_0000)
img_u8 = np.array(rhex_u8(f'{DATA_DIR}/test_images/img_0000.hex'), dtype=np.uint8)
golden_fc2 = np.array(
    [np.uint8(v).view(np.int8) for v in rhex_u8(f'{DATA_DIR}/test_images/img_0000_fc2.hex')],
    dtype=np.int8)
golden_label = int(open(f'{DATA_DIR}/test_images/img_0000_label.txt').read().strip())
golden_pred = int(open(f'{DATA_DIR}/test_images/img_0000_pred.txt').read().strip())

print(f'Golden: pred={golden_pred}, label={golden_label}')
print(f'Golden logits: {list(golden_fc2)}')

## 5. 结果比对

In [ ]:
hw_logits_i8 = np.array(logits, dtype=np.int32).astype(np.int8)
golden_logits_i32 = golden_fc2.astype(np.int32)

logit_match = np.array_equal(hw_logits_i8, golden_fc2)
pred_match = (pred == golden_pred)

print('=' * 50)
print('PicoRV32 CIM SoC Verification')
print('=' * 50)
print(f'  HW prediction:     {pred}')
print(f'  Python prediction:  {golden_pred}')
print(f'  True label:         {golden_label}')
print(f'  Prediction match:   {"YES" if pred_match else "NO"}')
print(f'  Logits bit-exact:   {"YES" if logit_match else "NO"}')
print()
if pred_match and logit_match:
    print('>>> PASS: PicoRV32 controls CIM correctly <<<')
    print('>>> RISC-V produces bit-exact same results as ARM/Python <<<')
else:
    print('>>> FAIL: Results mismatch <<<')
    if not logit_match:
        for i in range(10):
            if hw_logits_i8[i] != golden_fc2[i]:
                print(f'    logit[{i}]: hw={hw_logits_i8[i]}, golden={golden_fc2[i]}')
print('=' * 50)